# 🚧 UNLET-ADAS: Road Sign Detector Training (Colab, free GPU)
### B.E. Major Project | SJBIT Bengaluru | CSE 2025-26
**GitHub:** https://github.com/DEEK-SHITH/UNLET-ADAS

Trains a dedicated multi-class YOLOv8 road-sign detector (crosswalk /
speedlimit / stop / trafficlight). This is a **separate model** from
the main ADAS detector (person/car/bus/traffic light/stop sign) —
those come from COCO, which has no generic road-sign classes, so
warning/regulatory signs need their own model trained on a labeled
road-sign dataset. Same pattern as the pothole detector
(`UNLET_ADAS_Pothole_Colab.ipynb`).

Dataset: the classic 877-image andrewmvd "Road Sign Detection" dataset
(Kaggle, 2020), hosted on Roboflow Universe as one of Roboflow's own
curated Roboflow-100 benchmark datasets, `roboflow-100/road-signs-6ih4y`.

| Cell | What it does |
|---|---|
| 1 | Setup — install packages, mount Drive, clone GitHub |
| 2 | Configuration — Roboflow API key + all paths in one place |
| 3 | Download the road-sign dataset from Roboflow |
| 4 | Train YOLOv8 road-sign detector (~15–25 min on a T4 GPU) |
| 5 | Check results — metrics + sample predictions |
| 6 | Save weights + deployment instructions |

**Run cells top to bottom. Do not skip any cell.**

You need a free Roboflow API key for Cell 2 — sign up at
https://app.roboflow.com, then **Settings → API Keys**. Paste your own
key there; never share it publicly (if a key you pasted somewhere ever
becomes visible to others, regenerate it from that same Settings page).


In [ ]:
# ============================================================
# CELL 1 — Setup
# ============================================================

# Anti-disconnect — run this first
from IPython.display import display, Javascript
display(Javascript('''
function ClickConnect(){
    var btns = document.querySelectorAll("colab-toolbar-button");
    for(var i=0;i<btns.length;i++){
        if(btns[i].id=="connect") btns[i].click();
    }
}
setInterval(ClickConnect, 55000)
'''))
print('Anti-disconnect active!')

# Mount Google Drive (so trained weights survive when the Colab runtime recycles)
from google.colab import drive
drive.mount('/content/drive')

# Install packages
!pip install ultralytics roboflow -q

# Clone or update GitHub repo
import os
if not os.path.exists('/content/UNLET-ADAS'):
    !git clone https://github.com/DEEK-SHITH/UNLET-ADAS.git /content/UNLET-ADAS
    print('Repo cloned!')
else:
    !cd /content/UNLET-ADAS && git pull
    print('Repo updated!')

import sys
if '/content/UNLET-ADAS' not in sys.path:
    sys.path.insert(0, '/content/UNLET-ADAS')

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
else:
    print('No GPU detected — go to Runtime > Change runtime type > T4 GPU, then re-run this cell.')
print('Setup complete!')


In [ ]:
# ============================================================
# CELL 2 — Configuration
# All paths (and your Roboflow key) are defined here. Edit only this
# cell if paths or settings need to change.
# ============================================================

ROBOFLOW_API_KEY = 'PASTE_YOUR_OWN_KEY_HERE'   # app.roboflow.com -> Settings -> API Keys

DATASET_DIR = '/content/signs_dataset'                                # downloaded fresh each run (fast, small)
SAVE_DIR    = '/content/drive/MyDrive/UNLET_Project/checkpoints'      # persists across sessions
MODEL_SIZE  = 'n'          # n = fastest/smallest, matches the lightweight theme of this project
EPOCHS      = 100
BATCH_SIZE  = 16
IMAGE_SIZE  = 640
PATIENCE    = 20           # early stop if val mAP doesn't improve for this many epochs

os.makedirs(SAVE_DIR, exist_ok=True)

if ROBOFLOW_API_KEY == 'PASTE_YOUR_OWN_KEY_HERE':
    print('MISSING: set ROBOFLOW_API_KEY above before continuing.')
else:
    print('Configuration OK. Ready for Cell 3.')
print(f'Save dir : {SAVE_DIR}')


In [ ]:
# ============================================================
# CELL 3 — Download the road-sign dataset from Roboflow
# ============================================================

from src.train_signs import download_dataset

print('Downloading road-sign dataset from Roboflow...')
dataset_location = download_dataset(ROBOFLOW_API_KEY, DATASET_DIR)
data_yaml = os.path.join(dataset_location, 'data.yaml')

print(f'\nDataset ready at: {dataset_location}')
print(f'data.yaml       : {data_yaml}')
assert os.path.exists(data_yaml), 'data.yaml not found — check the download output above for errors.'


In [ ]:
# ============================================================
# CELL 4 — Train YOLOv8 road-sign detector (~15-25 min on a T4 GPU)
# ============================================================

from ultralytics import YOLO

model = YOLO(f'yolov8{MODEL_SIZE}.pt')
results = model.train(
    data=data_yaml,
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    patience=PATIENCE,
    project=SAVE_DIR,
    name='signs_run',
    exist_ok=True,
)

print('\nTraining complete!')


In [ ]:
# ============================================================
# CELL 5 — Check results: metrics + sample predictions
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

run_dir = os.path.join(SAVE_DIR, 'signs_run')

# Training curves (loss / mAP / precision / recall over epochs)
results_png = os.path.join(run_dir, 'results.png')
if os.path.exists(results_png):
    img = mpimg.imread(results_png)
    plt.figure(figsize=(14, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Training curves')
    plt.show()

# Sample validation predictions (model's own boxes drawn on val images)
val_pred = os.path.join(run_dir, 'val_batch0_pred.jpg')
if os.path.exists(val_pred):
    img = mpimg.imread(val_pred)
    plt.figure(figsize=(14, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Sample validation predictions')
    plt.show()

best_weights = os.path.join(run_dir, 'weights', 'best.pt')
print(f'\nBest weights: {best_weights}')
print(f'Exists      : {os.path.exists(best_weights)}')


In [ ]:
# ============================================================
# CELL 6 — Save weights + deployment instructions
# ============================================================

import shutil

best_weights = os.path.join(SAVE_DIR, 'signs_run', 'weights', 'best.pt')
final_path   = os.path.join(SAVE_DIR, 'signs_best.pt')

if os.path.exists(best_weights):
    shutil.copy(best_weights, final_path)
    print(f'Saved to Google Drive: {final_path}')
    print()
    print('To enable road sign detection in the Streamlit app:')
    print('  1. Download this file from your Google Drive.')
    print('  2. Rename it to: signs_best.pt')
    print('  3. Place it at: app/signs_best.pt   (in your local UNLET-ADAS clone)')
    print('  4. Restart the Streamlit app — the "Enable Sign Detection" checkbox')
    print('     will switch from disabled to available.')
else:
    print(f'Expected weights at {best_weights} but they were not found — '
          'check Cell 4\'s training log for errors.')
